## Data Preprocessing Techniques

- Data Cleaning
  - Data Imputation
  - Feature Scaling
- Feature Transformationss
  - Polynomial Features
  - Discretization
  - Handling categorical variables
  - Custom Transformers
  - Composite Transfromers
    - Apply Composite feature to diverse features
    - TargetTransformRegressor
  - Feature Selection
    - Filter based methods
    - Wrapper based methods
  - Feature extraction
    - PCA


These transfromations are applined in a specific order and the order can be specified via Pipeline.

## Importing basic libraries

In [21]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid")

## 1. Feature Extraction

### DictVectorizer
`DictVectorizer` converts a list of dictionary objects into feature matrix

In [5]:
data = [
   {'age': 4, 'height': 96.0}, 
   {'age': 1, 'height': 73.9}, 
   {'age': 3, 'height': 88.9}, 
   {'age': 2, 'height': 81.6} 
]

In [10]:
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse = False)
transformed_data = dv.fit_transform(data)
transformed_data, transformed_data.shape

(array([[ 4. , 96. ],
        [ 1. , 73.9],
        [ 3. , 88.9],
        [ 2. , 81.6]]),
 (4, 2))

## 2. Data Imputation

- Many ML algorithms need full feature matrix
- Data Imputation identifies missing values in each features and replaces with a strategy such as,
    - Mean/Median/Mode
    - User specified constant value

sklearn library provides `sklearn.impute.SimpleImputer` class for this purpose
`add_indicator` is a boolean parameter when set to `True` returns missing value indicators

In [49]:
cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg' , 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']
heart_data = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data', header=None, names=cols)
print(heart_data.shape)

(303, 14)


### STEP 1: Check if dataset contains missing values.

In [56]:
display(heart_data.info())
display(heart_data)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        299 non-null    object 
 12  thal      301 non-null    object 
 13  num       303 non-null    int64  
dtypes: float64(11), int64(1), object(2)
memory usage: 33.3+ KB


None

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0,1
299,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0,2
300,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0,3
301,57.0,0.0,2.0,130.0,236.0,0.0,2.0,174.0,0.0,0.0,2.0,1.0,3.0,1


In [26]:
heart_data.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
num         0
dtype: int64

In [44]:
print("Unique values in ca: ", heart_data.ca.unique())
print("Unique values in thal: ", heart_data.thal.unique())

Unique values in ca:  ['0.0' '3.0' '2.0' '1.0' '?']
Unique values in thal:  ['6.0' '3.0' '7.0' '?']


`?` missing values. Let's count the missing values

In [45]:
print("# Missing values in ca: ", heart_data.loc[heart_data.ca == '?', 'ca'].count())
print("# Missing values in thal: ", heart_data.loc[heart_data.thal == '?', 'thal'].count())

# Missing values in ca:  4
# Missing values in thal:  2


### STEP 2: Replace '?' with `nan`

In [51]:
heart_data.replace('?', np.nan, inplace=True)

### STEP3: Fill the `sklearn` missing values imputation utilities
here we use `SimpleImputer` with `mean` strategy

In [47]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(missing_values = np.nan, strategy = 'mean')
imputer = imputer.fit(heart_data)
heart_data_imputed = imputer.transform(heart_data)
print(heart_data_imputed.shape)

(303, 14)


`add_indicator = True` Adds additional column for earch column contain ing missing values.

In [58]:
imputer = SimpleImputer(strategy='mean', missing_values = np.nan, add_indicator = True)
imputer = imputer.fit(heart_data)
heart_data_imputed_with_indicator = imputer.transform(heart_data)
print(heart_data_imputed_with_indicator.shape)
display(heart_data_imputed_with_indicator[0])
display(heart_data)

(303, 16)


array([ 63. ,   1. ,   1. , 145. , 233. ,   1. ,   2. , 150. ,   0. ,
         2.3,   3. ,   0. ,   6. ,   0. ,   0. ,   0. ])

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0,1
299,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0,2
300,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0,3
301,57.0,0.0,2.0,130.0,236.0,0.0,2.0,174.0,0.0,0.0,2.0,1.0,3.0,1


## 3. Feature Scaling

Feature scaling transforms feature values such that all the features are on the same scale.

- Enables faster convergence in iterative optimization
- Algorithms which use euclidean distance between features also suffer
- Tree based ML algorithms are not affected by scaling - so feature scaling is not required

Feature scaling can be performed with the following methods:
- Standardization
- Normalization
- MaxAbsScaler

Dataset: [Abalone dataset](https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data)